In [ ]:
import logging
from medmnist import BreastMNIST
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import cv2
import random
from tmu.models.classification.vanilla_classifier import TMClassifier
from sklearn.metrics import roc_auc_score,  average_precision_score
from sklearn.metrics import classification_report
import json 
from time import time
import os
from sklearn.metrics import confusion_matrix
import seaborn as sns
from sklearn.metrics import roc_curve

In [ ]:
SEED = 42

np.random.seed(SEED)
random.seed(SEED)

In [ ]:
logging.getLogger("tmu").setLevel(logging.WARNING)
logging.getLogger("tmu.clause_bank.clause_bank_cuda").setLevel(logging.WARNING)
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

In [ ]:
def horizontal_flip(image):
    return cv2.flip(image, 1)

def rotate(image, angle):
    h, w = image.shape[:2]
    center = (w // 2, h // 2)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(image, matrix, (w, h), borderMode=cv2.BORDER_REFLECT)

def translate(image, tx, ty):
    h, w = image.shape[:2]
    matrix = np.float32([[1, 0, tx], [0, 1, ty]])
    return cv2.warpAffine(image, matrix, (w, h), borderMode=cv2.BORDER_REFLECT)

def adjust_brightness(image, factor):
    image = image.astype(np.float32)
    image = image * factor
    return np.clip(image, 0, 255).astype(np.uint8)

def adjust_contrast(image, factor):
    mean = np.mean(image, axis=(0, 1), keepdims=True)
    image = image.astype(np.float32)
    image = (image - mean) * factor + mean
    return np.clip(image, 0, 255).astype(np.uint8)

In [ ]:
def augment_image(image):
    if np.random.rand() < 0.5:
        image = horizontal_flip(image)
    if np.random.rand() < 0.5:
        image = rotate(image, np.random.uniform(-10, 10))
    if np.random.rand() < 0.5:
        image = translate(image, np.random.randint(-2, 3), np.random.randint(-2, 3))
    if np.random.rand() < 0.5:
        image = adjust_brightness(image, np.random.uniform(0.9, 1.1))
    if np.random.rand() < 0.5:
        image = adjust_contrast(image, np.random.uniform(0.9, 1.1))
    return image

In [ ]:
# Download dataset splits
train_dataset = BreastMNIST(split='train', download=True)
val_dataset   = BreastMNIST(split='val',   download=True)
test_dataset  = BreastMNIST(split='test',  download=True)

# Extract images and labels
X_train = train_dataset.imgs.astype(np.uint8)
Y_train = train_dataset.labels.flatten()

X_val   = val_dataset.imgs.astype(np.uint8)
Y_val   = val_dataset.labels.flatten()

X_test  = test_dataset.imgs.astype(np.uint8)
Y_test  = test_dataset.labels.flatten()

# Sanity checks
print("Image shapes:", X_train.shape, X_val.shape, X_test.shape)
print("Label shapes:", Y_train.shape, Y_val.shape, Y_test.shape)

# Class labels
labels = {
    0: "Malignant",  # Cancerous tumor
    1: "Benign"      # Non-cancerous tumor
}

print("Class mapping:", labels)


In [ ]:
def show_class_distribution(Y, split_name):
    unique, counts = np.unique(Y, return_counts=True)
    print(f"\nClass distribution in {split_name}:")
    for u, c in zip(unique, counts):
        print(f"  {u}: {labels[u]} — {c} samples")
    total = counts.sum()
    print(f"  Total samples: {total}")
    return pd.DataFrame({
        'class_id': unique,
        'label': [labels[u] for u in unique],
        'count': counts
    })

In [ ]:
def plot_class_histograms_horizontal(train_df, val_df, test_df):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)

    splits = [
        ("Train", train_df),
        ("Validation", val_df),
        ("Test", test_df),
    ]

    for ax, (name, df) in zip(axes, splits):
        ax.bar(df['label'], df['count'])
        ax.set_title(name)
        ax.set_xlabel("Class")
        ax.set_ylabel("Number of samples")

    fig.suptitle("Class distribution across dataset splits")
    fig.tight_layout()
    plt.show()


In [ ]:
train_counts = show_class_distribution(Y_train, "Train")
val_counts   = show_class_distribution(Y_val, "Validation")
test_counts  = show_class_distribution(Y_test, "Test")

plot_class_histograms_horizontal(train_counts, val_counts, test_counts)


In [ ]:
# Select one image from the training set
idx = 4
image = X_train[idx]
label = Y_train[idx]

# Apply augmentations
augmented_images = [
    ("Original", image),
    ("Horizontal flip", horizontal_flip(image)),
    ("Rotation (+10°)", rotate(image, angle=10)),
    ("Translation (2, -2)", translate(image, tx=2, ty=-2)),
    ("Brightness +10%", adjust_brightness(image, factor=1.1)),
    ("Contrast +10%", adjust_contrast(image, factor=1.1)),
]

# Plot all images horizontally
plt.figure(figsize=(14, 3))

for i, (title, img) in enumerate(augmented_images):
    plt.subplot(1, len(augmented_images), i + 1)
    plt.imshow(img, cmap="gray")   # 🔴 IMPORTANT FIX
    plt.title(title, fontsize=9)
    plt.axis("off")

plt.suptitle(f"Augmentation examples – {labels[label]}", fontsize=12)
plt.tight_layout()
plt.show()



In [ ]:
# 1. Identify minority class
minority_class = 0  # Malignant

# 2. Separate training data
X_min = X_train[Y_train == minority_class]
X_maj = X_train[Y_train != minority_class]

# 3. Augment minority class
X_min_aug = []
for img in X_min:
    for _ in range(2):  # 3–5x augmentation
        X_min_aug.append(augment_image(img))

# 4. Recombine training set
X_train_balanced = np.concatenate([X_maj, X_min, X_min_aug])
y_train_balanced = np.concatenate([
    Y_train[Y_train != minority_class],
    np.full(len(X_min), minority_class),
    np.full(len(X_min_aug), minority_class)
])


In [ ]:
train_counts = show_class_distribution(y_train_balanced, "Train")
val_counts   = show_class_distribution(Y_val, "Validation")
test_counts  = show_class_distribution(Y_test, "Test")

plot_class_histograms_horizontal(train_counts, val_counts, test_counts)


In [ ]:
for i in range(X_train_balanced.shape[0]):
            blur = cv2.GaussianBlur(X_train_balanced[i, :, :], (3, 3), 0)
            X_train_balanced[i, :, :] = np.where(cv2.Canny(blur, 100, 200) != 0, 1, 0)

for i in range(X_test.shape[0]):
            blur = cv2.GaussianBlur(X_test[i, :, :], (3, 3), 0)
            X_test[i, :, :] = np.where(cv2.Canny(blur, 100, 200) != 0, 1, 0)


for i in range(X_val.shape[0]): 
            blur = cv2.GaussianBlur(X_val[i, :, :], (3, 3), 0)
            X_val[i, :, :] = np.where(cv2.Canny(blur, 100, 200) != 0, 1, 0)


In [ ]:
X_train = X_train_balanced.astype(np.uint32)
X_val   = X_val.astype(np.uint32)
X_test  = X_test.astype(np.uint32)
Y_test  = Y_test.astype(np.uint32)
Y_train  = y_train_balanced.astype(np.uint32)
Y_val = Y_val.astype(np.uint32)

In [ ]:
epochs = 100
num_clauses = 200
T = 2000
s = 1.4329490174947763
patch_size = 1
max_included_literals = 18
weighted_best = True

In [ ]:
tm = TMClassifier(
        number_of_clauses=num_clauses,
        T=T,
        s=s,
        max_included_literals=max_included_literals,
        platform='GPU',
        weighted_clauses=weighted_best,
        patch_dim=(patch_size, patch_size),
    )


print(f"\nAccuracy over {epochs} epochs:\n")


if isinstance(labels, dict):
    labels = list(labels.values())
	
n_classes = len(np.unique(Y_test))


print("Labels:", labels)
reports_by_epoch = {}
os.makedirs("BreastMNIST/results_BreastMNIST_scores", exist_ok=True)

for epoch in range(epochs):
	start = time()
	tm.fit(X_train, Y_train, epochs=1, incremental=True)
	stop = time()
	Y_test_predicted, Y_test_scores=tm.predict(X_test, return_class_sums=True)
	result_test = 100*(Y_test_scores.argmax(axis=1) == Y_test).mean()
	
	print("#%d Accuracy: %.2f%% (%.2fs)" % (epoch+1, result_test, stop-start))
	
	
	y_true_bin = (Y_test == 0).astype(int)           # 1 if malignant, 0 if benign
	y_score_malignant = Y_test_scores[:, 0]          # score for malignant class

	# Binary AUC (macro and weighted are identical in binary problems)
	auc_macro = roc_auc_score(y_true_bin, y_score_malignant)
	print(Y_test_scores.shape)
	print(y_true_bin, y_score_malignant)
	auc_weighted = auc_macro

	report = classification_report(Y_test, Y_test_predicted, target_names=labels, output_dict=True)
	roc_auc_per_class = {
    labels[i]: float(roc_auc_score((Y_test == i).astype(int), Y_test_scores[:, i]))
	
    for i in range(n_classes)
}

	# Per-class PR-AUC (Average Precision)
	pr_auc_per_class = {
		labels[i]: float(average_precision_score((Y_test == i).astype(int), Y_test_scores[:, i]))
		for i in range(n_classes)
	}

	for name in labels:
		report[name]["roc_auc"] = roc_auc_per_class[name]
		report[name]["pr_auc"]  = pr_auc_per_class[name]
	
	report["roc_auc_macro_ovr"]    = float(auc_macro)
	report["roc_auc_weighted_ovr"] = float(auc_weighted)
	print(f"  ROC-AUC (macro)    = {auc_macro:.4f}")
	print(f"  ROC-AUC (weighted) = {auc_weighted:.4f}")
	reports_by_epoch[f"epoch_{epoch+1}"] = report
	


	df = pd.DataFrame(report).transpose()
	np.savetxt("BreastMNIST/results_BreastMNIST_scores/BreastMNIST_Augmented_Canny%d_%d_%d_%.1f_%d_%d_%d.txt" % (epoch+1, num_clauses, T, s, patch_size, max_included_literals, weighted_best), Y_test_scores, delimiter=',')



# with open("BreastMNIST/results_BreastMNIST_metrics/classification_Augmented_Canny_report.json", "w") as f:
# 		json.dump(reports_by_epoch, f, indent=4)

In [ ]:
Y_pred=tm.predict(X_test)
cm = confusion_matrix(Y_test, Y_pred, labels=np.arange(len(labels)))

plt.figure(figsize=(10, 8))
sns.set_theme(font_scale=1.0)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)

plt.title('Confusion Matrix (Final Epoch)', fontsize=16)
plt.xlabel('Predicted Class', fontsize=14)
plt.ylabel('True Class', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
_, Y_test_scores = tm.predict(X_test, return_class_sums=True)

# Define positive class (malignant = 0)
y_true_bin = (Y_test == 0).astype(int)
y_score_malignant = Y_test_scores[:, 0]

# Compute ROC curve
fpr, tpr, thresholds = roc_curve(y_true_bin, y_score_malignant)
auc_value = roc_auc_score(y_true_bin, y_score_malignant)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f'TM (AUC = {auc_value:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random')

plt.xlabel('False Positive Rate', fontsize=14)
plt.ylabel('True Positive Rate', fontsize=14)
plt.title('ROC Curve (Final Epoch)', fontsize=16)
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter

def extract_clauses_tmu(tm, class_id):
    print(tm.clause_banks[0].number_of_literals)
    n_features = tm.clause_banks[0].number_of_literals // 2
    clause_lengths = []
    feature_counter = Counter()

    n_clauses_per_polarity = tm.number_of_clauses // 2

    for polarity in (0, 1):  # positive + negative clauses
        for clause in range(n_clauses_per_polarity):
            clause_len = 0

            for ta in range(2 * n_features):
                if tm.get_ta_action(clause, ta, the_class=class_id, polarity=polarity):
                    clause_len += 1
                    feature_counter[ta % n_features] += 1

            if clause_len > 0:
                clause_lengths.append(clause_len)

    return clause_lengths, feature_counter

In [ ]:
import matplotlib.pyplot as plt

def plot_feature_counter_bar(feature_counter, title="Feature usage frequency"):
    features = list(feature_counter.keys())
    counts = list(feature_counter.values())

    plt.figure(figsize=(10, 4))
    plt.bar(features, counts)
    plt.xlabel("Feature index")
    plt.ylabel("Number of clauses using feature")
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

def plot_feature_frequency_boxplot(feature_counter, title="Feature frequency distribution"):
    frequencies = list(feature_counter.values())

    plt.figure()
    plt.boxplot(frequencies, vert=True)
    plt.ylabel("Number of clauses using feature")
    plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
clause_lengths, feature_counter = extract_clauses_tmu(tm, class_id=0)
print(f"\nExtracted {len(clause_lengths)} clauses for class 'Malignant' (0).")
print(f"Feature counter: {feature_counter}")
print(f"Mean clause length: {np.mean(clause_lengths):.2f}")
print(f"Median clause length: {np.median(clause_lengths):.2f}")
print(f"Min clause length: {np.min(clause_lengths):.2f}")
print(f"Max clause length: {np.max(clause_lengths):.2f}")

plot_feature_counter_bar(feature_counter, title="Feature usage frequency for class 'Malignant' (0)")

In [ ]:
plot_feature_frequency_boxplot(
    feature_counter,
    title="Feature frequency – TM-S1"
)


In [ ]:
def plot_clause_length_histogram(clause_lengths, specialist_name):
    plt.figure()
    plt.hist(clause_lengths, bins=np.arange(1, max(clause_lengths) + 2))
    plt.xlabel("Clause length (number of literals)")
    plt.ylabel("Number of clauses")
    plt.title(f"Clause length distribution – {specialist_name}")
    plt.tight_layout()
    plt.show()


In [ ]:
distribution = plot_clause_length_histogram(clause_lengths, "Test")


In [ ]:
len(clause_lengths)
